In [1]:
import sys
sys.path.append("..")

import time
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss

from src.features import (
    build_features, get_numeric_features, CATEGORICAL_FEATURES, BINARY_FEATURES,
)
from src.models import build_logistic_pipeline

train = pd.read_parquet("../data/processed/train.parquet")
valid = pd.read_parquet("../data/processed/valid.parquet")

X_train = train.drop(columns=["target"])
y_train = train["target"]
X_valid = valid.drop(columns=["target"])
y_valid = valid["target"]

numeric_features = get_numeric_features(build_features(train.head(1000)))

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Binary features:", len(BINARY_FEATURES))

Numeric features: 57
Categorical features: 4
Binary features: 1


In [2]:
pipeline = build_logistic_pipeline(numeric_features, CATEGORICAL_FEATURES, BINARY_FEATURES)

start = time.time()
pipeline.fit(X_train, y_train)
print(f"Training time: {time.time() - start:.0f} seconds")

Training time: 48 seconds


In [3]:
p_train = pipeline.predict_proba(X_train)[:, 1]
p_valid = pipeline.predict_proba(X_valid)[:, 1]

auc_train = roc_auc_score(y_train, p_train)
auc_valid = roc_auc_score(y_valid, p_valid)

print(f"Train AUC:      {auc_train:.4f}")
print(f"Validation AUC: {auc_valid:.4f}   Gini: {2 * auc_valid - 1:.4f}")
print(f"Validation Brier score: {brier_score_loss(y_valid, p_valid):.4f}")
print(f"Average predicted probability: {p_valid.mean():.4f}")
print(f"Actual default rate:           {y_valid.mean():.4f}")

Train AUC:      0.6761
Validation AUC: 0.6804   Gini: 0.3608
Validation Brier score: 0.1220
Average predicted probability: 0.1371
Actual default rate:           0.1512


In [4]:
grade_order = {g: i for i, g in enumerate("ABCDEFG", start=1)}
sub_grade_order = {
    f"{g}{n}": (i - 1) * 5 + n
    for i, g in enumerate("ABCDEFG", start=1)
    for n in range(1, 6)
}

auc_grade = roc_auc_score(y_valid, valid["grade"].map(grade_order))
auc_sub_grade = roc_auc_score(y_valid, valid["sub_grade"].map(sub_grade_order))

results = pd.DataFrame({
    "validation_auc": [auc_grade, auc_sub_grade, auc_valid],
}, index=["Lending Club grade", "Lending Club sub_grade", "Logistic regression"])
results["gini"] = 2 * results["validation_auc"] - 1
results.round(4)

,validation_auc,gini
Lending Club grade,0.6640,0.3280
Lending Club sub_grade,0.6750,0.3501
Logistic regression,0.6804,0.3608


In [5]:
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
coefficients = pipeline.named_steps["model"].coef_[0]

coef_table = (
    pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
)

print("Top 15 features by influence:")
coef_table.head(15)

Top 15 features by influence:


,feature,coefficient
63,num__missingindicator_pub_rec_bankruptcies,0.570842
94,num__missingindicator_percent_bc_gt_75,-0.558861
116,cat__purpose_small_business,0.544377
164,cat__addr_state_WV,-0.466128
71,num__missingindicator_bc_util,0.448945
108,cat__purpose_credit_card,-0.445912
60,num__missingindicator_revol_util,0.413313
100,cat__home_ownership_MORTGAGE,-0.391685
141,cat__addr_state_MS,0.382896
144,cat__addr_state_NH,-0.381516


In [6]:
joblib.dump(pipeline, "../models/logistic_baseline.joblib")
print("Saved.")

Saved.


In [7]:
numeric_only = coef_table[
    coef_table["feature"].str.startswith("num__")
    & ~coef_table["feature"].str.contains("missingindicator")
]
numeric_only.head(10)

,feature,coefficient
52,num__fico_avg,-0.215876
20,num__tot_cur_bal,-0.159009
2,num__dti,0.157986
55,num__loan_to_income,0.140596
24,num__bc_open_to_buy,-0.124800
27,num__mo_sin_old_rev_tl_op,-0.119772
22,num__acc_open_past_24mths,0.110554
31,num__mths_since_recent_bc,-0.104371
50,num__total_bc_limit,-0.103891
1,num__annual_inc,-0.099126
